# S2UT pilot with real-data diagnostics

Use fixed train and dev subsets for CUDA checks, T2TT training, alignment diagnostics, three strong S2UT conditions, and teacher-free dev evaluation.

| Setting | Default |
|---|---|
| Train / dev | Up to 10,000 / 200 pairs in the original splits |
| T2TT / S2UT | 20,000 / 10,000 updates |
| End of annealing | 5,000 updates, with beta zero for the remainder |
| Conditions | Baseline / Annealed Monotonic Guidance / Annealed T2TT Guidance |

This standalone pilot writes directly to Drive. For the main comparison and ZIP recovery, use the [native training workflow](../../docs/BENCHMARK_COMPARISON.md).

Use a GPU runtime. The corpus is read-only and the pilot uses train and dev. Rerun with the same settings and `ROOT` to restore the last checkpoint. [Detailed workflow](https://github.com/Yaaamashiro/alignment-transfer-s2ut/blob/main/docs/PILOT.md)

## GitHub access

For a private repository, create a fine-grained personal access token in GitHub **Settings → Developer settings → Personal access tokens → Fine-grained tokens**. Select the repository owner, choose **Only select repositories**, select `alignment-transfer-s2ut`, and grant **Contents → Read-only**. Set an expiration date. See the [GitHub token guide](https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/managing-your-personal-access-tokens).

In Colab, open **Secrets** (the key icon), add `GITHUB_TOKEN`, paste the token into its value, and enable **Notebook access**. The setup cell reads this secret for Git operations. Public repositories also work without a token. Successful clone/fetch and checkout in the setup cell confirm access to the requested revision.

To authenticate an additional repository, add its exact HTTPS URL to `GITHUB_AUTH_REPOS` in the settings cell and grant the same token read access to that repository. The benchmark is public and uses anonymous HTTPS by default.

After a runtime reset, use the same notebook revision, experiment settings, and Drive paths, then rerun the cells. To open the notebook after the repository becomes private, download the `.ipynb` while signed into GitHub and use **File → Upload notebook** in Colab, or open a copy saved in your Drive. Enable secret access for that notebook. The runtime token is used after the notebook opens.


## 1. Mount Drive and configure the pilot

Set the subset sizes, updates, and output directory. Use a new `ROOT` for changed training settings.

In [ ]:
#@title Process logging and progress
from pathlib import Path
import json
"""Standard-library-only process logging, also embedded in Colab before install."""


def log_event(name, state, logs, detail=""):
    from datetime import datetime, timezone
    from pathlib import Path
    root = Path(logs)
    root.mkdir(parents=True, exist_ok=True)
    line = f"{datetime.now(timezone.utc).isoformat()} [{name}] {state} {detail}\n"
    with (root / "notebook-events.log").open("a", encoding="utf-8") as stream:
        stream.write(line)
        stream.flush()
    print(line, end="", flush=True)


def graceful_sigterm(function):
    """Turn a first POSIX SIGTERM into the ordinary interruption cleanup path."""
    from functools import wraps
    import os
    import signal
    import threading

    @wraps(function)
    def wrapped(*args, **kwargs):
        if (os.name != "posix" or threading.current_thread() is not threading.main_thread()
                or signal.getsignal(signal.SIGTERM) != signal.SIG_DFL):
            return function(*args, **kwargs)
        interrupted = False

        def interrupt(signum, frame):
            nonlocal interrupted
            if not interrupted:
                interrupted = True
                raise KeyboardInterrupt("SIGTERM")

        previous = signal.signal(signal.SIGTERM, interrupt)
        try:
            return function(*args, **kwargs)
        finally:
            signal.signal(signal.SIGTERM, previous)
    return wrapped


@graceful_sigterm
def run_process(command, logs, *, name="command", capture_json=False, env=None, cwd=None,
                heartbeat_seconds=10., extra_record=None):
    """Stream every child line; silence triggers liveness, never fake progress.

    stdout JSON is kept apart from stderr. Nested loggers share an owned POSIX
    process group so interruption cannot strand a second training process.
    """
    from datetime import datetime, timezone
    import hashlib
    import json
    import os
    from pathlib import Path
    import queue
    import signal
    import subprocess
    import tempfile
    import threading
    import time

    if heartbeat_seconds <= 0:
        raise ValueError("heartbeat_seconds must be positive")
    root = Path(logs)
    root.mkdir(parents=True, exist_ok=True)
    command = list(map(str, command))
    key = hashlib.sha256(json.dumps(name, ensure_ascii=False).encode()).hexdigest()[:12] + "-" + str(time.time_ns())
    output, result_path, receipt = root / (key + ".log"), root / (key + ".stdout.log"), root / (key + ".json")
    launch_env = dict(os.environ if env is None else env)
    launch_env["PYTHONUNBUFFERED"] = "1"
    termination_timeout = 75 if launch_env.get("AT_LOG_WANDB_DRAIN") == "1" else 5
    owns_group = os.name == "posix" and launch_env.get("AT_LOG_MANAGED_GROUP") != "1"
    if owns_group:
        launch_env["AT_LOG_MANAGED_GROUP"] = "1"
    started, clock = datetime.now(timezone.utc).isoformat(), time.perf_counter()
    record = dict(name=name, command=command, started_utc=started,
        host_utc_offset_seconds=int(datetime.now().astimezone().utcoffset().total_seconds()),
        elapsed_seconds=0., returncode=None, status="running", exception=None,
        pid=None, log=str(output.resolve()), stdout_log=str(result_path.resolve()) if capture_json else None,
        gpu_csv=None, gpu_monitor_error=None, cwd=str(Path(cwd or Path.cwd()).resolve()))
    lock, stop, done = threading.Lock(), threading.Event(), threading.Event()
    messages, reader_errors = queue.Queue(maxsize=256), []
    process = reader = result_stream = None
    stream = output.open("w", encoding="utf-8")

    def save_record():
        record["elapsed_seconds"] = time.perf_counter() - clock
        fd, temporary = tempfile.mkstemp(dir=root, prefix=".log-record-")
        try:
            with os.fdopen(fd, "w", encoding="utf-8") as target:
                json.dump(record, target, ensure_ascii=False, indent=2, allow_nan=False)
                target.write("\n")
                target.flush()
            os.replace(temporary, receipt)
        finally:
            if os.path.exists(temporary):
                os.unlink(temporary)

    def notice(state, detail=""):
        line = f"[{name}] {state} pid={record['pid']} elapsed={time.perf_counter()-clock:.1f}s {detail}\n"
        with lock:
            stream.write(line)
            stream.flush()
        print(line, end="", flush=True)

    def read_lines(pipe):
        try:
            for line in pipe:
                with lock:
                    stream.write(line)
                    stream.flush()
                while not stop.is_set():
                    try:
                        messages.put(line, timeout=.1)
                        break
                    except queue.Full:
                        pass
        except Exception as error:
            reader_errors.append(error)
        finally:
            done.set()

    def terminate_child():
        if process is None:
            return
        if owns_group:
            try:
                os.killpg(process.pid, signal.SIGTERM)
            except ProcessLookupError:
                pass
        elif process.poll() is None:
            process.terminate()
        try:
            process.wait(timeout=termination_timeout)
        except subprocess.TimeoutExpired:
            if owns_group:
                os.killpg(process.pid, signal.SIGKILL)
            else:
                process.kill()
            process.wait()
        if owns_group:
            # A nested child may outlive its exited leader after cancellation.
            try:
                os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass

    try:
        notice("started", f"full log: {output.resolve()}")
        save_record()
        if capture_json:
            result_stream = result_path.open("w", encoding="utf-8")
        process = subprocess.Popen(command, stdout=result_stream if capture_json else subprocess.PIPE,
            stderr=subprocess.PIPE if capture_json else subprocess.STDOUT, text=True,
            encoding="utf-8", errors="replace", bufsize=1, env=launch_env, cwd=cwd,
            start_new_session=owns_group)
        record["pid"] = process.pid
        notice("running", "waiting for child output")
        save_record()
        pipe = process.stderr if capture_json else process.stdout
        reader = threading.Thread(target=read_lines, args=(pipe,), daemon=True, name="command-log-reader")
        reader.start()
        last_output = last_notice = time.perf_counter()
        while not (done.is_set() and messages.empty() and process.poll() is not None):
            try:
                line = messages.get(timeout=min(.1, heartbeat_seconds))
            except queue.Empty:
                line = None
            if line is not None:
                print(line, end="", flush=True)
                if line.strip():
                    last_output = time.perf_counter()
            if reader_errors:
                raise reader_errors[0]
            now = time.perf_counter()
            if now - max(last_output, last_notice) >= heartbeat_seconds:
                notice("running", f"no_output={now-last_output:.1f}s (liveness only; progress not confirmed)")
                save_record()
                last_notice = now
        code = process.wait()
        if reader_errors:
            raise reader_errors[0]
        if capture_json and code == 0:
            json.loads(result_path.read_text(encoding="utf-8"))
        record.update(returncode=code, status="succeeded" if code == 0 else "failed_or_interrupted")
        if code:
            raise subprocess.CalledProcessError(code, command)
        notice("completed", f"exit={code}; full log: {output.resolve()}")
    except BaseException as error:
        record.update(status="failed_or_interrupted", exception=type(error).__name__ + ": " + str(error))
        terminate_child()
        if process is not None:
            record["returncode"] = process.returncode
        notice("interrupted" if isinstance(error, KeyboardInterrupt) else "failed",
               f"exit={record['returncode']} {type(error).__name__}; full log: {output.resolve()}")
        raise
    finally:
        stop.set()
        if reader is not None:
            reader.join(timeout=2)
        if process is not None and (reader is None or not reader.is_alive()):
            pipe = process.stderr if capture_json else process.stdout
            if pipe is not None:
                pipe.close()
        if result_stream is not None:
            result_stream.close()
        if extra_record is not None:
            record.update(extra_record() if callable(extra_record) else extra_record)
        save_record()
        stream.close()
    result = subprocess.CompletedProcess(command, record["returncode"],
        result_path.read_text(encoding="utf-8") if capture_json else None)
    result.record = record
    return result


"""Standard-library GitHub authentication, embedded before Colab installation."""


GITHUB_GIT_WORKER = r'''
import base64
import os
import subprocess
import sys
from urllib.parse import quote

redactions = []
for i in range(int(os.environ.get("GIT_CONFIG_COUNT", "0"))):
    value = os.environ.get(f"GIT_CONFIG_VALUE_{i}", "")
    if value.startswith("Authorization: Basic "):
        encoded = value.split()[-1]
        token = base64.b64decode(encoded).decode("utf-8").split(":", 1)[1]
        redactions.extend((value, encoded, token, quote(token, safe="")))
redactions = sorted(set(redactions), key=len, reverse=True)
process = None
try:
    command = sys.argv[1:]
    position = 1
    while position < len(command) and command[position] == "-C":
        position += 2
    if command[position] in {"fetch", "pull"}:
        remote = next((arg for arg in command[position + 1:] if not arg.startswith("-")), "origin")
        if "://" not in remote:
            checked = subprocess.run([*command[:position], "remote", "get-url", "--all", remote],
                stdin=subprocess.DEVNULL, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                text=True, encoding="utf-8", errors="replace")
            from urllib.parse import urlsplit
            for address in checked.stdout.splitlines():
                url = urlsplit(address)
                if url.username or url.password or url.query or url.fragment:
                    raise ValueError("unsafe stored remote")
    process = subprocess.Popen(command, stdin=subprocess.DEVNULL,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        encoding="utf-8", errors="replace", bufsize=1)
    for line in process.stdout:
        for secret in redactions:
            line = line.replace(secret, "[REDACTED]")
        print(line, end="", flush=True)
    code = process.wait()
except Exception:
    print("Git could not complete the requested operation.", file=sys.stderr, flush=True)
    code = 1
finally:
    if process is not None:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
        process.stdout.close()
if code:
    print("Check the repository URL and revision. For a private repository, add "
          "GITHUB_TOKEN to Colab Secrets, enable notebook access, and grant the "
          "fine-grained token Contents: Read-only for the selected repository. "
          "Check token expiry and any organization approval.", file=sys.stderr, flush=True)
sys.exit(code)
'''


def run_github_git(command, logs, *, runner, repositories, name="git", env=None, cwd=None,
                   capture_json=False):
    """Authenticate network Git calls with URL-scoped, process-only headers."""
    import base64
    import os
    from pathlib import Path
    import re
    import sys
    from urllib.parse import urlsplit

    command = list(map(str, command))
    position = 1
    while position < len(command) and command[position] == "-C":
        position += 2
    operation = command[position] if position < len(command) else ""
    if operation not in {"clone", "fetch", "pull", "ls-remote"}:
        return runner(command, logs, name=name, env=env, cwd=cwd, capture_json=capture_json)
    if Path(command[0]).name.lower() not in {"git", "git.exe"}:
        raise ValueError("GitHub authentication requires a git command")
    for argument in command:
        if "://" in argument:
            url = urlsplit(argument)
            if url.username or url.password or url.query or url.fragment:
                raise ValueError("Use repository URLs without credentials, queries, or fragments")
    scopes = []
    for repository in repositories:
        url = urlsplit(str(repository))
        if (url.scheme != "https" or url.netloc != "github.com" or url.query or url.fragment
                or not re.fullmatch(r"/[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", url.path)):
            raise ValueError("Authentication scopes must be plain HTTPS GitHub repository URLs")
        scopes.append(url.geturl())

    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        pass  # Public repositories remain accessible when Secrets are unavailable.
    if token is not None and (not isinstance(token, str) or any(ord(c) < 33 or ord(c) > 126 for c in token)):
        token = None
        raise ValueError("GITHUB_TOKEN must contain a valid GitHub token without whitespace")
    git_env = dict(os.environ if env is None else env)
    for key in list(git_env):
        if (key.startswith(("GIT_TRACE", "GIT_CONFIG_KEY_", "GIT_CONFIG_VALUE_"))
                or key in {"GIT_CONFIG_COUNT", "GIT_CONFIG_PARAMETERS", "GIT_CURL_VERBOSE", "GITHUB_TOKEN", "GH_TOKEN"}):
            git_env.pop(key)
    git_env.update(GIT_TERMINAL_PROMPT="0", GCM_INTERACTIVE="Never", GIT_ASKPASS="", GIT_TRACE_REDACT="1")
    settings = [("credential.helper", ""), ("http.extraHeader", ""), ("http.followRedirects", "false")]
    header = None
    try:
        if token:
            header = "Authorization: Basic " + base64.b64encode(("x-access-token:" + token).encode()).decode()
            settings.extend(("http." + scope + ".extraHeader", header) for scope in dict.fromkeys(scopes))
        git_env["GIT_CONFIG_COUNT"] = str(len(settings))
        for i, (key, value) in enumerate(settings):
            git_env[f"GIT_CONFIG_KEY_{i}"] = key
            git_env[f"GIT_CONFIG_VALUE_{i}"] = value
        return runner([sys.executable, "-c", GITHUB_GIT_WORKER, *command], logs,
                      name=name, env=git_env, cwd=cwd, capture_json=capture_json)
    except Exception:
        raise RuntimeError("GitHub access failed. Check the logged URL/revision error and Colab Secrets "
                           "GITHUB_TOKEN access, selected repositories, Contents: Read-only, and expiry.") from None
    finally:
        token = header = None
        settings.clear()
        git_env.clear()


def execute(*arguments, name=None, capture_json=False, env=None, cwd=None):
    command = list(map(str, arguments))
    if name is None:
        name = command[2].rsplit('.', 1)[-1] if len(command) > 2 and command[1] == '-m' else Path(command[0]).name
        if '--method' in command:
            name += ':' + command[command.index('--method') + 1]
        if '--prior-type' in command:
            name += ':' + command[command.index('--prior-type') + 1]
    if command and command[0] == 'git':
        repositories = [globals().get('REPO_URL', 'https://github.com/Yaaamashiro/alignment-transfer-s2ut.git'),
                        *globals().get('GITHUB_AUTH_REPOS', [])]
        return run_github_git(command, LOGS, runner=run_process, repositories=repositories,
                              name=name, capture_json=capture_json, env=env, cwd=cwd)
    return run_process(command, LOGS, name=name, capture_json=capture_json, env=env, cwd=cwd)


def execute_json(*arguments, name=None, env=None, cwd=None):
    return json.loads(execute(*arguments, name=name, capture_json=True, env=env, cwd=cwd).stdout)

LOGS = Path('/content/alignment-transfer-logs/pilot/bootstrap')


In [ ]:
from google.colab import drive
log_event('drive-mount', 'started', LOGS)
try:
    drive.mount('/content/drive')
except BaseException as error:
    log_event('drive-mount', 'failed', LOGS, type(error).__name__)
    raise
else:
    log_event('drive-mount', 'completed', LOGS)
import json, os, subprocess, sys, time
from pathlib import Path


In [ ]:
#@title Inputs and outputs
REPO_URL = 'https://github.com/Yaaamashiro/alignment-transfer-s2ut.git'
REPO_REVISION = '107605f4f5bd7b53722d73ced3fa5a76560e7a9d'
CORPUS = '/content/drive/MyDrive/ja-en-direct-s2st-corpus-data' #@param {type:"string"}
ACCEPTED = CORPUS + '/production/manifests/releases/accepted.jsonl'
ROOT = '/content/drive/MyDrive/alignment-transfer-s2ut-data/pilot-text-001' #@param {type:"string"}


In [ ]:
#@title Subset size, training, and evaluation
TRAIN_LIMIT = 10000 #@param {type:"integer"}
DEV_LIMIT = 200 #@param {type:"integer"}
SEED = 1 #@param {type:"integer"}
BATCH_SIZE = 8 #@param {type:"integer"}
UPDATE_FREQUENCY = 4 #@param {type:"integer"}
MAX_TOKENS = 20000 #@param {type:"integer"}
LOADER_WORKERS = 2 #@param {type:"integer"}
PREFETCH_BATCHES = 2 #@param {type:"integer"}
LOCAL_CACHE = '/content/alignment-transfer-cache' #@param {type:"string"}
CACHE_GB = 8.0 #@param {type:"number"}
T2TT_UPDATES = 20000 #@param {type:"integer"}
S2UT_UPDATES = 10000 #@param {type:"integer"}
ANNEAL_STEPS = 5000 #@param {type:"integer"}
RUN_CUDA_INTEGRATION = True #@param {type:"boolean"}
GENERATE_WAVEFORMS = False #@param {type:"boolean"}
ASR_BLEU = False #@param {type:"boolean"}


In [ ]:
#@title Working paths and configuration checks
PYTHON = '/content/alignment-venv/bin/python'
REPO = Path('/content/alignment-transfer-s2ut')
DATA = Path(ROOT) / 'data'
MANIFEST = DATA / 'manifest.jsonl'
RUNS = Path(ROOT) / 'runs'
LOGS = Path('/content/alignment-transfer-logs/pilot') / Path(ROOT).name
CONDITIONS = ['baseline', 'annealed_monotonic', 'annealed_t2tt']
assert not Path(ROOT).resolve().is_relative_to(Path(CORPUS).resolve())
assert Path(ACCEPTED).is_file(), ACCEPTED
assert 1 <= ANNEAL_STEPS < S2UT_UPDATES
assert not ASR_BLEU or GENERATE_WAVEFORMS, 'ASR-BLEU requires generated waveforms'
os.environ['HF_HOME'] = '/content/huggingface'
os.environ['MPLBACKEND'] = 'Agg'
os.environ['PYTHONUNBUFFERED'] = '1'
os.environ['MPLCONFIGDIR'] = '/content/matplotlib-cache'


## 2. Set up and check the environment

Install Python 3.10.22, torch and torchaudio 2.7.1, and pinned fairseq. Use a new `ROOT` when changing code, seed, or data.

Commands record startup, PID, live output, and completion or failure in `LOGS`. After ten seconds of silence, the logger reports liveness.


In [ ]:
#@title Set up Python and fairseq
if not (REPO / '.git').is_dir():
    execute('git', 'clone', '--no-checkout', REPO_URL, str(REPO), name='clone-alignment')
execute('git', '-C', str(REPO), 'fetch', 'origin', REPO_REVISION, name='fetch-alignment')
execute('git', '-C', str(REPO), 'checkout', '--detach', REPO_REVISION, name='checkout-alignment')
os.chdir(REPO)
execute(sys.executable, '-m', 'pip', 'install', 'uv', name='install-uv')
execute('uv', 'venv', '--python', '3.10.22', '--seed', '--allow-existing', '/content/alignment-venv', name='create-python-environment')
for packages in [
    ['pip==23.3.2', 'setuptools<81', 'wheel'],
    ['torch==2.7.1', 'torchaudio==2.7.1', 'numpy==1.26.4', 'Cython==3.2.9'],
    ['--no-build-isolation', '-r', 'requirements-fairseq.txt'],
    ['-r', 'requirements.txt'],
]:
    execute(PYTHON, '-m', 'pip', 'install', *packages)
execute(PYTHON, '-m', 'pip', 'check', name='dependency-check')


In [ ]:
def run(name, module, *arguments, gpu=False, env=None):
    command = [PYTHON, '-m', 'alignment_transfer_s2ut.pilot', 'run', '--name', name, '--logs', str(LOGS)]
    if gpu:
        command.append('--monitor-gpu')
    command += ['--', PYTHON, '-m', module, *map(str, arguments)]
    return execute(*command, name='notebook:' + name, env=env)


In [ ]:
run('runtime', 'alignment_transfer_s2ut.pilot', 'runtime')
runtime = execute_json(PYTHON, '-m', 'alignment_transfer_s2ut.pilot', 'runtime', name='runtime-details')
(Path(ROOT) / 'runtime.json').write_text(json.dumps(runtime, indent=2), encoding='utf-8')
print(runtime)
run('cpu-tests', 'pytest', '-v', '--tb=short', '-m', 'not gpu')


## 3. Select a fixed pilot manifest

Sample up to 10,000 train and 200 dev pairs within the original splits using a fixed seed. Configure limits above. Hash selected audio only. Save the selection, source manifest hash, and population counts and check them on resume.

In [ ]:
run('prepare-data', 'alignment_transfer_s2ut.pilot', 'prepare', '--input', ACCEPTED,
    '--corpus-root', CORPUS, '--output', DATA, '--train-limit', TRAIN_LIMIT, '--dev-limit', DEV_LIMIT,
    '--seed', SEED, '--resume')
selection = json.loads((DATA / 'selection.json').read_text())
print({key: selection[key] for key in ['population_counts', 'selected_counts', 'test_used', 'manifest_hash']})
rows = [json.loads(line) for line in MANIFEST.read_text().splitlines() if line.strip()]
run('architecture', 'alignment_transfer_s2ut.compare_architectures', '--t2tt-config', 'configs/t2tt/fisher.yaml',
    '--s2ut-config', 'configs/s2ut/fisher.yaml', '--output', Path(ROOT) / 'architecture.json', '--resume')


## 4. CUDA integration with real models

Use up to two selected train pairs and one dev pair to check real CTC, HuBERT, the three strong conditions, and the official vocoder. This two-update check saves each attempt separately.

In [ ]:
if RUN_CUDA_INTEGRATION:
    tiny = [row for row in rows if row['split'] == 'train'][:2] + [row for row in rows if row['split'] == 'dev'][:1]
    integration = Path(ROOT) / 'integration'
    integration.mkdir(parents=True, exist_ok=True)
    tiny_manifest = integration / 'input.jsonl'
    tiny_manifest.write_text(''.join(json.dumps(row, ensure_ascii=False) + '\n' for row in tiny), encoding='utf-8')
    environment = os.environ.copy()
    environment['AT_S2UT_GPU_MANIFEST'] = str(tiny_manifest)
    run('cuda-integration', 'pytest', 'tests/test_external_pipeline.py', '-m', 'gpu', '-q', '-s',
        '--basetemp', integration / ('attempt-' + str(time.time_ns())), gpu=True, env=environment)
else:
    log_event('cuda-integration', 'skipped', LOGS, 'RUN_CUDA_INTEGRATION=False')


## 5. T2TT, alignment, and units

Train T2TT on the same pilot train pairs using its own update budget. The integration teacher has a separate destination. Generate all dev translations from source text with the best dev-loss checkpoint and measure BLEU. Review BLEU, EOS rate, sources, predictions, and references before preprocessing.

In [ ]:
run('t2tt', 'alignment_transfer_s2ut.train_t2tt', '--manifest', MANIFEST, '--config', 'configs/t2tt/fisher.yaml',
    '--output', Path(ROOT) / 't2tt', '--max-updates', T2TT_UPDATES, '--warmup-updates', 1000,
    '--seed', SEED, '--batch-size', BATCH_SIZE, '--update-frequency', UPDATE_FREQUENCY, '--max-tokens', MAX_TOKENS,
    '--loader-workers', LOADER_WORKERS, '--prefetch-batches', PREFETCH_BATCHES, '--pin-memory',
    '--validation-interval', 500, '--save-interval-updates', 500, '--device', 'cuda', '--resume-if-exists', gpu=True)
run('t2tt-dev-text', 'alignment_transfer_s2ut.evaluate_t2tt', '--manifest', MANIFEST, '--split', 'dev',
    '--checkpoint', Path(ROOT) / 't2tt/checkpoint_best.pt', '--beam-size', 5, '--max-chars', 1000,
    '--length-penalty', 1.0, '--output', Path(ROOT) / 't2tt/dev-text-metrics.json', '--device', 'cuda', '--resume', gpu=True)
text_metrics = json.loads((Path(ROOT) / 't2tt/dev-text-metrics.json').read_text())
print({key: text_metrics[key] for key in ['text_bleu', 'bleu_signature', 'samples', 'eos_reached_ratio', 'brevity_penalty']})
text_predictions = [json.loads(line) for line in (Path(ROOT) / 't2tt/dev-text-metrics-predictions.jsonl').read_text().splitlines() if line.strip()]
dev_text_by_id = {row['pair_id']: row['en_text'] for row in rows if row['split'] == 'dev'}
for prediction in text_predictions[:5]:
    print('JA:', prediction['source_text'], '\nHYP:', prediction['hypothesis'], '\nREF:', dev_text_by_id[prediction['pair_id']])
run('mt-attention', 'alignment_transfer_s2ut.extract_t2tt_attention', '--manifest', MANIFEST,
    '--checkpoint', Path(ROOT) / 't2tt/checkpoint_best.pt', '--output', Path(ROOT) / 'mt', '--device', 'cuda', '--resume', gpu=True)
run('ctc', 'alignment_transfer_s2ut.align_speech_text', '--manifest', MANIFEST, '--config', 'configs/alignment/ctc.yaml',
    '--output', Path(ROOT) / 'ctc', '--device', 'cuda', '--resume', gpu=True)
run('km100', 'alignment_transfer_s2ut.fetch_assets', '--config', 'configs/units/hubert_km100.yaml')
run('units', 'alignment_transfer_s2ut.extract_units', '--manifest', MANIFEST, '--config', 'configs/units/hubert_km100.yaml',
    '--output', Path(ROOT) / 'units', '--device', 'cuda', '--resume', gpu=True)
run('teacher', 'alignment_transfer_s2ut.build_teacher_alignment', '--manifest', MANIFEST,
    '--mt-attention', Path(ROOT) / 'mt', '--alignment', Path(ROOT) / 'ctc', '--units', Path(ROOT) / 'units',
    '--config', 'configs/s2ut/fisher.yaml', '--output', Path(ROOT) / 'teacher', '--resume')


## 6. Train and dev alignment diagnostics

Save up to 500 samples and 20 heatmaps per split. Inspect coverage, invalid rows, and raw and smoothed entropy. Compare the CTC spans with audio and original text to assess timing.

In [ ]:
from IPython.display import display, Image, Audio
by_id = {row['pair_id']: row for row in rows}
for split in ['train', 'dev']:
    audit = Path(ROOT) / ('alignment-audit-' + split)
    run('audit-' + split, 'alignment_transfer_s2ut.diagnose_alignment', '--manifest', MANIFEST,
        '--teacher', Path(ROOT) / 'teacher', '--units', Path(ROOT) / 'units', '--config', 'configs/s2ut/fisher.yaml',
        '--split', split, '--max-samples', 500, '--heatmaps', 20, '--seed', SEED, '--source-mapping', 'frontend',
        '--prior-smoothing', 0.05, '--output', audit, '--overwrite')
    summary = json.loads((audit / 'summary.json').read_text())
    print(split, {key: summary[key] for key in ['audited_pairs', 'valid_source_tokens_ratio',
        'valid_target_tokens_ratio', 'valid_unit_rows_ratio', 'prior_entropy', 'smoothed_entropy', 'zero_cell_ratio']})
    assert summary['counts']['valid_unit_rows'] > 0, 'No usable alignment rows; inspect CTC and teacher first'
    pairs = [json.loads(line) for line in (audit / 'pairs.jsonl').read_text().splitlines() if line.strip()]
    # Display two examples per split; all 20 PNG/NPZ pairs remain in the audit directory.
    import hashlib
    for pair in pairs[:2]:
        row = by_id[pair['pair_id']]
        filename = hashlib.sha256(json.dumps(row['pair_id'], ensure_ascii=False, sort_keys=True, separators=(',', ':')).encode()).hexdigest()
        print(row['pair_id'], row['ja_text'], '/', row['en_text'])
        display(Image(filename=str(audit / 'heatmaps' / (filename + '.png'))))
        display(Audio(filename=row['ja_audio']), Audio(filename=row['en_audio']))


## 7. Three strong conditions

Use the same initialization, data, batch settings, learning rate, 1,000-update warmup, and update budget for all conditions. With 10,000 updates, annealing ends at 5,000 and beta stays at zero afterward. The plan cell saves the planned commands. Training resumes from checkpoints and stops on failure.

Training starts with the first ready inputs and CPU workers prefetch later inputs. The three conditions share a bounded local cache for unaugmented FBANK/CMVN, units, and teacher artifacts. The training thread applies SpecAugment before GPU transfer.

In [ ]:
PILOT_ARGS = ['--manifest', str(MANIFEST), '--config', 'configs/s2ut/fisher.yaml', '--units', str(Path(ROOT) / 'units'),
    '--teacher', str(Path(ROOT) / 'teacher'), '--output', str(RUNS), '--logs', str(LOGS),
    '--max-updates', str(S2UT_UPDATES), '--anneal-steps', str(ANNEAL_STEPS), '--seed', str(SEED),
    '--batch-size', str(BATCH_SIZE), '--update-frequency', str(UPDATE_FREQUENCY), '--max-tokens', str(MAX_TOKENS), '--loader-workers', str(LOADER_WORKERS),
    '--prefetch-batches', str(PREFETCH_BATCHES), '--cache-dir', LOCAL_CACHE, '--cache-gb', str(CACHE_GB)]
plan = execute(PYTHON, '-m', 'alignment_transfer_s2ut.pilot', 'plan', *PILOT_ARGS, name='pilot-plan', capture_json=True).stdout
(Path(ROOT) / 'pilot-plan.json').write_text(plan, encoding='utf-8')
print(plan)


### Run training

Review the plan before running. Train the three conditions sequentially and resume each from its own checkpoint.

In [ ]:
execute(PYTHON, '-m', 'alignment_transfer_s2ut.pilot', 'train', *PILOT_ARGS)
run('comparison', 'alignment_transfer_s2ut.compare_experiments', *[RUNS / name for name in CONDITIONS],
    '--output', Path(ROOT) / 'fair-comparison.json')


## 8. Shared dev evaluation and reporting

Evaluate all dev pairs with beam 5, at most 1,000 units, and beta zero. The default metric is unit error rate. Set `GENERATE_WAVEFORMS` and `ASR_BLEU` to enable the common pinned vocoder and ASR. Use dev for pilot tuning and evaluate test separately.

In [ ]:
speech_only = Path(ROOT) / 'dev-speech-only.jsonl'
run('dev-input', 'alignment_transfer_s2ut.prepare_inference_input', '--manifest', MANIFEST,
    '--split', 'dev', '--output', speech_only, '--resume')
if GENERATE_WAVEFORMS:
    run('vocoder', 'alignment_transfer_s2ut.fetch_assets', '--config', 'configs/s2ut/vocoder.yaml')
for name in CONDITIONS:
    folder = RUNS / name
    vocoder = ['--vocoder-config', 'configs/s2ut/vocoder.yaml'] if GENERATE_WAVEFORMS else ['--units-only']
    run(name + '-dev-infer', 'alignment_transfer_s2ut.infer', '--input', speech_only,
        '--checkpoint', folder / 'checkpoint_best.pt', '--beam-size', 5, '--max-units', 1000,
        '--device', 'cuda', '--output', folder / 'dev-predictions.jsonl', '--resume', *vocoder, gpu=True)
    asr = ['--asr-config', 'configs/experiments/evaluation.yaml', '--device', 'cuda'] if ASR_BLEU else []
    run(name + '-dev-evaluate', 'alignment_transfer_s2ut.evaluate', '--manifest', MANIFEST, '--split', 'dev',
        '--predictions', folder / 'dev-predictions.jsonl', '--units', Path(ROOT) / 'units',
        '--output', folder / 'dev-metrics.json', '--resume', *asr, gpu=ASR_BLEU)
run('pilot-report', 'alignment_transfer_s2ut.pilot', 'report', '--runs', RUNS, '--logs', LOGS,
    '--t2tt-evaluation', Path(ROOT) / 't2tt/dev-text-metrics.json', '--output', Path(ROOT) / 'pilot-summary.json')
summary = json.loads((Path(ROOT) / 'pilot-summary.json').read_text())
display(summary)
display(Image(filename=str(Path(ROOT) / 'pilot-summary.png')))

if not GENERATE_WAVEFORMS:
    log_event('waveform-generation', 'skipped', LOGS, 'GENERATE_WAVEFORMS=False')
if not ASR_BLEU:
    log_event('asr-bleu', 'skipped', LOGS, 'ASR_BLEU=False')


## 9. Saved results

- `data/selection.json` stores the manifest hash, selection settings, original split counts, and selected counts.
- `t2tt/dev-text-metrics.json` and `t2tt/dev-text-metrics-predictions.jsonl` store dev BLEU, the SacreBLEU signature, EOS rate, generation settings, and predictions.
- `alignment-audit-train/` and `alignment-audit-dev/` store coverage, entropy, per-sample metrics, and up to 20 heatmaps per split.
- `runs/{condition}/` contains checkpoints, training metrics, dev predictions, and scores.
- VM-local `LOGS` contains commands, statuses, elapsed times, full output, and GPU utilization and memory sampled every five seconds.
- `runtime.json` records Python, torch, CUDA, fairseq revision, GPU model, and memory.
- `fair-comparison.json`, `pilot-summary.json`, and `pilot-summary.png` record comparison checks, best dev loss, beta, update counts, dev scores, timings, and curves.

On failure, check `LOGS` and resume the cell with the same settings. Match GPU utilization to training logs to check for FBANK or I/O stalls. Assess corpus results after running the pilot.

`input/data_wait_seconds` measures input waiting time. `input/optimization_seconds` measures CPU augmentation, GPU transfer, forward and backward passes, and optimization. Validation and saves are excluded. `LOCAL_CACHE` is VM-local and rebuilt on demand.